# Coding Assistant Evaluations: Introduction and Setup
This module shows how to build evals for coding / software development related usecases. It explains the problem and an approach. Then it shows the task format and how the tasks run. At the end, you do the setup for the module.

## 1. What Is an Eval for a Coding Agent?

An eval is a set of examples where you already know the correct answer. You run the agent on each example and compare its output to the answer. The result is a score that you can track over time. If you are new to evals, start with the [Foundational Evaluations](../../Foundational%20Evaluations/) modules.

For a chatbot, an example is a prompt and an expected response. For a coding agent, an example can be much larger. It could contain a full repository, a task in that repository, and the correct result.

This module evaluates two types of coding agents:

- A **PR review agent** reads a change and reports the defects. Then it approves the change or requests changes.
- A **coding agent** reads an issue and writes a patch that fixes it.

## 2. The Problem

Two problems make coding evals more difficult than other evals.

### Problem 1: The examples should come from your own code

Public benchmarks, for example SWE-bench and [Code Review Bench](https://github.com/withmartian/code-review-benchmark), use PRs from popular open-source projects. These benchmarks test for general defects, for example race conditions and missing null checks. A high score shows that the agent can find general defects. It does not show that the agent knows the rules of your team.

Here is an example. A coding agent opens a PR that moves a plan ID from Stripe metadata into your database. The agent does not know that a different billing service reads that metadata at each renewal. The tests for each service pass. One month later, the renewals reset customers to their old plans.

A generic review agent approves this PR, because the change looks like a good cleanup. A reviewer who knows your system blocks it. Public benchmarks do not contain defects of this type, because they do not know how your services connect. To measure this skill, you need examples from your own code.

### Problem 2: Each example needs a full, sealed environment

A review agent cannot review a diff alone. It must read the code around the change, and it must sometimes build the code and run the tests. Thus each example needs the repository at the correct commit, with all dependencies installed. Think about the time that a new developer needs to run your code. Now do that again for each example.

The environment must also hide the answer. The correct fix is in a later commit, and the review comments are on GitHub. Agents search for data when they can. If the agent finds the answer, the score measures search, not review.

## 3. The Approach

### Use your git history as the dataset

You already have a labeled dataset. Each PR in your repository records the code before the change, the change, what the reviewer said, and the fix that the reviewer approved. Your team did the labeling work during normal code review.

| Item | Where git keeps it | Use in the eval |
|------|--------------------|-----------------|
| The code before the change | The parent commit | The start state for the agent |
| The change | The diff from the parent commit to the PR commit | The input for a PR review agent |
| The reviewer decision | The PR review comments and the verdict | The answer for a PR review agent |
| The correct fix | The merged patch and its tests | The answer for a coding agent |

Each example is a commit hash and a PR number. Thus you can make the same example again at any time. Also keep the finished examples in a git repository. Then each version of your eval has a commit, and you can compare scores across versions.

### Roll back, seal, and compare to the oracle

For each PR, the approach has three steps:

1. **Roll back the repository to the parent commit of the change.** The agent sees the code as the reviewer saw it, before the fix. If you start after the fix, the defect is not there.
2. **Seal the environment.** Remove all paths to the answer. Then the score measures the skill of the agent, not its ability to search.
3. **Use the PR review or the merged patch as the oracle.** The oracle is the gold standard answer. The scorer compares the agent output to the oracle.

```
parent commit ──► PR change ──► review ──────────► fix ──► merge
      ▲               │            │                         │
  roll back here    input      oracle for a            oracle for a
                               PR review agent         coding agent
```

### How to seal the environment

A seal has three layers. Each layer closes one path to the answer.

| Layer | Path to the answer | Control |
|-------|--------------------|---------|
| **Git** | `git log --all` shows the fix commit | Clone only the parent commit. Remove the remote, the remote refs, and the reflog. |
| **Network** | `gh pr view` shows the review comments | Allow only the LLM API and the package registries. Do not allow GitHub. |
| **Verifier** | The agent reads or changes the oracle | Keep the oracle in a separate verifier environment. The agent cannot access it. |

### Why this approach solves the problem

| Problem | Solution |
|---------|----------|
| Public benchmarks do not know your code. | The examples come from your PR history. They contain the rules of your team. |
| Manual environments for each example take too much time. | A Dockerfile rolls back the repository and installs the dependencies. A tool builds it for each example. |
| The agent can find the answer. | The seal closes the git, network, and verifier paths. |

## 4. The Harbor Task Format

Each example should be in a standard format. Then a tool can run any agent on any example, and other people can run your eval again and get the same result.

This module uses the task format from [Harbor](https://github.com/harbor-framework/harbor). Harbor is not necessary for this approach. But its format keeps the agent files and the verifier files separate, and many benchmarks and tutorials already use it. it also has tooling that's useful to run these eval datapoints

In Harbor, each example is one **task directory**:

```text
tasks/pr-1234-reject/
├── task.toml          # agent and verifier settings
├── instruction.md     # the instruction that the agent reads
├── environment/
│   └── Dockerfile     # rolls back to the parent commit, seals git, applies the change
├── tests/             # verifier only
│   ├── test.sh        # verifier entry point, always writes a reward file
│   ├── score.py       # compares the agent output to the oracle
│   └── oracle.json    # the oracle
└── solution/
    └── solve.sh       # submits the oracle, to test the scorer
```

The most important part of this format is the split between the agent and the verifier:

- **The agent sees** `task.toml`, `instruction.md`, and the container that `environment/` builds. The instruction is the same for all tasks. It does not refer to the PR or the answer.
- **Only the verifier sees** `tests/` and `solution/`. Harbor never puts these files into the agent container.

`task.toml` also sets the network seal. Harbor sets `network_mode` to `public` by default, so you must change it to `allowlist`:

```toml
[environment]
network_mode = "allowlist"
allowed_hosts = ["llm-api.example.com", "pypi.org", "*.pythonhosted.org"]  # never github.com

[verifier]
environment_mode = "separate"
```

## 5. How the Tasks Run

A **trial** is one agent on one task. Harbor runs each trial in a new container and deletes the container after the trial. Thus each trial starts from the same state, and trials cannot change each other.

```
tasks/ ──► Harbor ──► one trial for each task and agent configuration
                        ├─ start a container from the task image
                        ├─ the agent does the task (network: allowlist only)
                        ├─ copy the agent output to the verifier
                        └─ reward.json ──► scoreboard
```

Before you can trust a score, each task must build and run correctly. Do these checks for each task:

1. **Build the image.** The Dockerfile must clone the parent commit, install the dependencies, seal git, and apply the change. A failure here costs no LLM tokens.
2. **Run the oracle.** Harbor submits the oracle as the agent output. The score must be perfect. If it is not perfect, the scorer has a defect.
3. **Run the agent.** Only now does the score tell you about the agent.

In this workshop, you run Docker on your computer. This is sufficient for a small number of tasks. In practice, run the trials on a compute substrate, for example Amazon EC2 or Lambda MicroVMs. Then you can run many trials at the same time, and each trial has isolation and a fixed network policy.

> **Note:** Some microVM platforms give full internet access or no access. On these platforms, send the traffic through a VPC with a firewall. Allow only the hosts in the allowlist.

## 5. Next Notebooks

| Notebook | Content |
|----------|---------|
| **02 PR agent eval** | Examine two prepared PR review datapoints from [awslabs/mcp](https://github.com/awslabs/mcp). Select the metrics. Run Codex as a PR review agent. |
| **03 coding agent eval** | Coming soon: evaluate a coding agent that writes the fix. |

## 6. Setup

| Tool | Purpose |
|------|---------|
| **Docker** | Builds and runs the sealed environment for each task |
| **Harbor** | Reads the tasks, runs the trials, and runs the verifiers |
| **Codex CLI** | Makes the datapoints. It is also the agent under test. |
| **GitHub CLI** (`gh`) | Reads PRs and review comments to make the datapoints |
| **git** | Clones the repositories |

> **Note:** Install Docker first. All tasks run in containers.
> - macOS or Windows: install [Docker Desktop](https://docs.docker.com/get-docker/).
> - Linux: install [Docker Engine](https://docs.docker.com/engine/install/).

Harbor needs Python 3.12 or later.

In [ ]:
%pip install -q -r requirements.txt

An editor that you open from the Dock or Finder does not read your shell profile, so the kernel cannot find `docker`, `gh`, `codex`, or `harbor`. This cell adds your shell `PATH` to the kernel.

The cell also loads your settings from `.env`. To use a named AWS profile or a different Region, copy `.env.example` to `.env` and set `AWS_PROFILE` and `AWS_REGION`. Then restart the kernel or run this cell again.

In [ ]:
from env_setup import fix_path, load_env; fix_path(); load_env()

### Check your `.env` file

The notebooks read `AWS_REGION` and `AWS_PROFILE` from `.env`. If `.env` does not exist, this cell makes it from `.env.example`. Then it shows the settings that the notebooks use.

If you do not have default AWS credentials, set `AWS_PROFILE` in `.env` to a profile in `~/.aws/config`. At Amazon, this is usually your Isengard profile. Then run this cell again.

In [ ]:
import os
import shutil
from pathlib import Path

import boto3
import botocore.session

if not Path(".env").exists():
    shutil.copy(".env.example", ".env")
    print("Made .env from .env.example. Open .env and set AWS_PROFILE if you need it.")
load_env()  # load .env again, so changes that you make are used

region = os.environ.get("AWS_REGION", "us-east-1")
profile = os.environ.get("AWS_PROFILE")
print(f"AWS_REGION:  {region}")
print(f"AWS_PROFILE: {profile or 'not set (default credential chain)'}")

if profile and profile not in botocore.session.Session().available_profiles:
    raise RuntimeError(f"AWS_PROFILE={profile} is not in ~/.aws/config. Correct it in .env, then run this cell again.")
if boto3.Session(profile_name=profile).get_credentials() is None:
    raise RuntimeError("boto3 found no AWS credentials. Set AWS_PROFILE in .env, then run this cell again.")
print("AWS credentials found.")

### Do the Setup Checks

This cell examines each prerequisite. It also makes sure that the Docker daemon runs. If a check fails, the cell tells you what to do.

In [ ]:
import shutil
import subprocess
import sys

DOCKER_INSTALL_MSG = """
Docker is necessary to run the tasks in this module.
Install Docker, then run this cell again:
  - macOS or Windows: https://docs.docker.com/get-docker/  (Docker Desktop)
  - Linux:            https://docs.docker.com/engine/install/
"""

def run(cmd, timeout=30):
    """Run a command. Return (ok, output)."""
    try:
        # stdin=DEVNULL: `codex exec` reads stdin until EOF, and the kernel stdin never closes.
        out = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout,
                             stdin=subprocess.DEVNULL)
        return out.returncode == 0, (out.stdout + out.stderr).strip()
    except (FileNotFoundError, subprocess.TimeoutExpired) as e:
        return False, str(e)

def check_python():
    ok = sys.version_info >= (3, 12)
    return ok, f"Python {sys.version.split()[0]}" + ("" if ok else " (Harbor needs 3.12 or later)")

def check_docker_installed():
    if not shutil.which("docker"):
        return False, "docker is not on PATH." + DOCKER_INSTALL_MSG
    return run(["docker", "--version"])

def check_docker_running():
    if not shutil.which("docker"):
        return False, "skipped, Docker is not installed"
    ok, out = run(["docker", "info", "--format", "{{.ServerVersion}}"])
    if ok:
        return True, f"daemon runs, server {out}"
    return False, "The Docker daemon does not run. Start Docker Desktop (or `sudo systemctl start docker`). Then run this cell again."

def check_cli(name, version_args, install_hint):
    if not shutil.which(name):
        return False, f"{name} is not on PATH. {install_hint}"
    ok, out = run([name, *version_args])
    return ok, out.splitlines()[0] if out else ""

def check_gh_auth():
    if not shutil.which("gh"):
        return False, "skipped, gh is not installed"
    ok, _ = run(["gh", "auth", "status"])
    return ok, "logged in" if ok else "Run `gh auth login`."

checks = {
    "Python 3.12+":     check_python(),
    "Docker installed": check_docker_installed(),
    "Docker running":   check_docker_running(),
    "git":              check_cli("git", ["--version"], "Install git: https://git-scm.com/downloads"),
    "GitHub CLI":       check_cli("gh", ["--version"], "Install gh: https://cli.github.com/"),
    "GitHub CLI auth":  check_gh_auth(),
    "Codex CLI":        check_cli("codex", ["--version"], "Install Codex: `npm install -g @openai/codex`"),
    "Harbor":           check_cli("harbor", ["--version"], "Install Harbor: `pip install harbor`"),
}

for name, (ok, detail) in checks.items():
    print(f"{'OK ' if ok else 'ERR'} {name:<18} {detail}")

failed = [name for name, (ok, _) in checks.items() if not ok]
if failed:
    raise RuntimeError(f"Repair these items, then run this cell again: {', '.join(failed)}")
print("\nAll checks passed.")

### Configure Codex

Codex has two roles in this module:

1. **Collector.** Codex runs on your computer. It reads PRs with `gh` and writes the task directories.
2. **Agent under test.** Codex runs in the sealed container. It cannot access GitHub.

Log in to Codex with your model provider. Then run the cell below.

<!-- TODO: document the provider configuration for the workshop (API key, or Bedrock / custom provider in ~/.codex/config.toml) -->

In [ ]:
# Run Codex on Amazon Bedrock. These flags are the same as putting this in ~/.codex/config.toml:
#   model = "openai.gpt-5.5"
#   model_provider = "amazon-bedrock"
#   [model_providers.amazon-bedrock.aws]
#   region = "us-east-1"
# Passing them on the command line keeps the setup inside the project, so you don't edit your global config.
CODEX_MODEL = "openai.gpt-6-sol"
import os
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")  # from .env, if you set it
CODEX_ARGS = [
    "--model", CODEX_MODEL,
    "-c", 'model_provider="amazon-bedrock"',
    "-c", f'model_providers.amazon-bedrock.aws.region="{AWS_REGION}"',
]

ok, out = run(["codex", "exec", "--skip-git-repo-check", *CODEX_ARGS, "Reply with the single word: hello"],
               timeout=180)  # first Codex start-up can be slow
print(out[-500:])
assert ok, "Codex did not complete the prompt on Bedrock. Check your AWS credentials and Bedrock model access in AWS_REGION."

## Summary

- Your git history contains your eval data.
- Roll back to the parent commit. Seal the environment. Use the PR review or the merged patch as the oracle.
- The Harbor task format keeps the agent files and the verifier files separate.
- In this workshop, tasks run on local Docker. In practice, run them on a compute substrate.

**Next:** [02 PR agent eval](02%20PR%20agent%20eval.ipynb)